In [1]:
import pandas as pd
import requests
import os
import urllib3
import json

# SSL 경고 무시
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# 1. 설정 및 데이터 로드
file_path = r'C:\Users\Kunny\Documents\GitHub\CAGI\EvoStructCLIP\Stability\S2450.tsv'
save_dir = 'S2450_pdb_files'
os.makedirs(save_dir, exist_ok=True)

df = pd.read_csv(file_path, sep='\t')
unique_ids = df['PDB'].unique()

def get_pdb_info(acc_id):
    """ID 형태에 따라 PDB ID와 Chain을 분리하고 정보를 가져옴"""
    acc_id = str(acc_id).strip()
    
    # 케이스 1: 1fh5H (4자 PDB + 1자 Chain)
    if len(acc_id) == 5:
        return acc_id[:4].lower(), acc_id[4:].upper()

    # 케이스 2: 1n2a (순수 4자 PDB)
    elif len(acc_id) == 4:
        return acc_id.lower(), "Unknown"

    # 케이스 3: UniProt ID 또는 혼합형 (앞 6글자 추출)
    else:
        uniprot_id = acc_id[:6]
        url = f"https://www.ebi.ac.uk/pdbe/api/mappings/best_structures/{uniprot_id}"
        try:
            response = requests.get(url, verify=False, timeout=10)
            if response.status_code == 200:
                data = response.json()
                if uniprot_id in data:
                    best = data[uniprot_id][0]
                    return best['pdb_id'], best['chain_id']
        except:
            pass
    return None, None

def download_structure(pdb_id):
    """PDB 다운로드 시도 후 실패 시 CIF 다운로드"""
    pdb_id = pdb_id.lower()
    
    # 1. PDB 시도
    pdb_url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    pdb_path = os.path.join(save_dir, f"{pdb_id}.pdb")
    
    if os.path.exists(pdb_path):
        return pdb_path, "PDB"

    try:
        r = requests.get(pdb_url, verify=False, timeout=10)
        if r.status_code == 200:
            with open(pdb_path, 'wb') as f:
                f.write(r.content)
            return pdb_path, "PDB"
    except:
        pass

    # 2. CIF 시도 (PDB 없을 경우)
    cif_url = f"https://files.rcsb.org/download/{pdb_id}.cif"
    cif_path = os.path.join(save_dir, f"{pdb_id}.cif")
    
    try:
        r = requests.get(cif_url, verify=False, timeout=10)
        if r.status_code == 200:
            with open(cif_path, 'wb') as f:
                f.write(r.content)
            return cif_path, "CIF"
    except:
        pass

    return None, None

# 2. 실행 루프
mapping_results = {}
total = len(unique_ids)

print(f"총 {total}개의 고유 ID에 대해 수집을 시작합니다.\n" + "="*50)

for i, acc_id in enumerate(unique_ids, 1):
    print(f"[{i}/{total}] 처리 중: {acc_id}...", end=" ")
    
    pdb_id, chain_id = get_pdb_info(acc_id)
    
    if pdb_id:
        path, file_type = download_structure(pdb_id)
        if path:
            mapping_results[acc_id] = {
                "pdb_id": pdb_id,
                "chain": chain_id,
                "file": path,
                "type": file_type
            }
            print(f"성공 ({pdb_id.upper()}, Chain: {chain_id}, Type: {file_type})")
        else:
            print(f"❌ 다운로드 실패 ({pdb_id})")
    else:
        print(f"⚠️ 정보 없음")

# 3. JSON 저장
with open('mapping_results_2450.json', 'w', encoding='utf-8') as f:
    json.dump(mapping_results, f, indent=4)

print("="*50 + f"\n[완료] mapping_results.json 저장됨. (총 {len(mapping_results)}개 수집)")

총 115개의 고유 ID에 대해 수집을 시작합니다.
[1/115] 처리 중: P03050... 성공 (1BAZ, Chain: A, Type: PDB)
[2/115] 처리 중: P96110... 성공 (1B26, Chain: A, Type: PDB)
[3/115] 처리 중: P00648... 성공 (6PQK, Chain: A, Type: PDB)
[4/115] 처리 중: P13726... 성공 (9P0X, Chain: T, Type: PDB)
[5/115] 처리 중: P41016... 성공 (1C9O, Chain: A, Type: PDB)
[6/115] 처리 중: P32081... 성공 (3PF4, Chain: A, Type: PDB)
[7/115] 처리 중: P00282... 성공 (8F5L, Chain: A, Type: PDB)
[8/115] 처리 중: P0A3H0... 성공 (1HUU, Chain: A, Type: PDB)
[9/115] 처리 중: P13479... 성공 (1FR2, Chain: A, Type: PDB)
[10/115] 처리 중: P06396... 성공 (6Q9R, Chain: A, Type: PDB)
[11/115] 처리 중: P05798... 성공 (1LNI, Chain: A, Type: PDB)
[12/115] 처리 중: P06876... 성공 (1H89, Chain: C, Type: PDB)
[13/115] 처리 중: P30289... 성공 (1MGR, Chain: A, Type: PDB)
[14/115] 처리 중: P0A9X9... 성공 (2L15, Chain: A, Type: PDB)
[15/115] 처리 중: P19614... 성공 (3QF6, Chain: A, Type: PDB)
[16/115] 처리 중: P07445... 성공 (6C1X, Chain: A, Type: PDB)
[17/115] 처리 중: P22069... 성공 (3SNF, Chain: A, Type: PDB)
[18/115] 처리 중: P00651... 성공 

In [2]:
import pandas as pd
import json
import os
from Bio.PDB import PDBParser, MMCIFParser, Polypeptide

# 1. 데이터 및 JSON 로드
with open('mapping_results_2450.json', 'r') as f:
    mapping_results = json.load(f)

df = pd.read_csv(file_path, sep='\t')

# 새 결과 컬럼 추가
df['PDB_ID_MAPPED'] = None
df['PDB_CHAIN_MAPPED'] = None
df['PDB_POS_MAPPED'] = None
df['PDB_WT_MAPPED'] = None
df['MAPPING_STATUS'] = "Failed"

def get_res_wt(chain_obj, pos):
    """특정 체인과 위치에서 아미노산 1글자 표기 추출"""
    try:
        res = chain_obj[pos]
        if Polypeptide.is_aa(res):
            return Polypeptide.three_to_one(res.get_resname())
    except:
        return None
    return None

print("인덱스 검증 및 매핑 시작...")

for idx, row in df.iterrows():
    orig_id = str(row['PDB']).strip()
    if orig_id not in mapping_results:
        continue
        
    info = mapping_results[orig_id]
    pdb_path = info['file']
    target_chain = info['chain']
    target_pos = int(row['POS'])
    target_wt = row['WT']
    
    # 파서 선택
    parser = MMCIFParser(QUIET=True) if info['type'] == "CIF" else PDBParser(QUIET=True)
    
    try:
        struct = parser.get_structure('temp', pdb_path)
        model = struct[0]
        
        # 1. 지정된 체인 확인 (없으면 첫 번째 체인)
        if target_chain == "Unknown" or target_chain not in model:
            actual_chain_id = list(model.get_chains())[0].id
        else:
            actual_chain_id = target_chain
            
        chain_obj = model[actual_chain_id]
        
        # 2. 해당 위치 아미노산 체크
        pdb_wt = get_res_wt(chain_obj, target_pos)
        
        # 3. 결과 기록
        df.at[idx, 'PDB_ID_MAPPED'] = info['pdb_id']
        df.at[idx, 'PDB_CHAIN_MAPPED'] = actual_chain_id
        
        if pdb_wt == target_wt:
            df.at[idx, 'PDB_POS_MAPPED'] = target_pos
            df.at[idx, 'PDB_WT_MAPPED'] = pdb_wt
            df.at[idx, 'MAPPING_STATUS'] = "Success"
        else:
            # WT가 다르면 근처(+/- 10)를 뒤져서 맞는 번호가 있는지 확인 (Off-set 보정)
            found_offset = False
            for offset in range(-10, 11):
                if offset == 0: continue
                alt_wt = get_res_wt(chain_obj, target_pos + offset)
                if alt_wt == target_wt:
                    df.at[idx, 'PDB_POS_MAPPED'] = target_pos + offset
                    df.at[idx, 'PDB_WT_MAPPED'] = alt_wt
                    df.at[idx, 'MAPPING_STATUS'] = f"Offset_{offset}"
                    found_offset = True
                    break
            
            if not found_offset:
                df.at[idx, 'PDB_WT_MAPPED'] = pdb_wt
                df.at[idx, 'MAPPING_STATUS'] = "WT_Mismatch"

    except Exception as e:
        df.at[idx, 'MAPPING_STATUS'] = f"Error: {str(e)}"

# 결과 요약
print("\n" + "="*30)
print(df['MAPPING_STATUS'].value_counts())
print("="*30)

# 최종 데이터 저장
output_name = "S2450_final_mapped_coords.tsv"
df.to_csv(output_name, sep='\t', index=False)
print(f"\n[완료] 매핑 결과가 {output_name}에 저장되었습니다.")

인덱스 검증 및 매핑 시작...

MAPPING_STATUS
Success        972
WT_Mismatch    763
Offset_-1      162
Offset_-10      62
Offset_-8       49
Offset_-3       48
Offset_-7       47
Offset_-4       47
Offset_-6       46
Offset_-5       42
Offset_-9       41
Offset_1        34
Offset_-2       31
Offset_4        20
Offset_5        17
Offset_2        13
Offset_10       13
Offset_6        10
Offset_3         9
Offset_7         8
Offset_9         7
Offset_8         7
Failed           2
Name: count, dtype: int64

[완료] 매핑 결과가 S2450_final_mapped_coords.tsv에 저장되었습니다.


In [3]:
from Bio import SeqIO, pairwise2
import pandas as pd
import json
import os
from Bio.PDB import PDBParser, MMCIFParser, Polypeptide

# 1. FASTA 파일 로드 (딕셔너리 형태로 저장)
fasta_path = r"C:\Users\Kunny\Documents\GitHub\CAGI\EvoStructCLIP\Stability\S2450.fasta"
fasta_sequences = {record.id: str(record.seq) for record in SeqIO.parse(fasta_path, "fasta")}

# 2. 결과 저장을 위한 데이터 로드
with open('mapping_results_2450.json', 'r') as f:
    mapping_results = json.load(f)
df = pd.read_csv("S2450_final_mapped_coords.tsv", sep='\t')

def get_pdb_seq_and_map(chain_obj):
    """PDB 체인에서 서열과 각 잔기의 실제 번호(Residue ID) 리스트 추출"""
    pdb_seq = ""
    pdb_res_ids = []
    for res in chain_obj:
        if Polypeptide.is_aa(res):
            pdb_seq += Polypeptide.three_to_one(res.get_resname())
            pdb_res_ids.append(res.get_id()[1])
    return pdb_seq, pdb_res_ids

print("FASTA 기반 정밀 Alignment 매핑 시작...")

for idx, row in df.iterrows():
    # 이미 Success인 경우는 건너뛰고 싶다면 조건 추가 가능 (여기서는 전체 재검토 권장)
    orig_id = str(row['PDB']).strip()
    # FASTA ID 찾기 (데이터셋의 PDB 열 값이 FASTA 헤더와 일치한다고 가정)
    # 만약 P11532WTK18N 형태라면 앞 6글자만 사용
    fasta_id = orig_id if orig_id in fasta_sequences else orig_id[:6]
    
    if fasta_id not in fasta_sequences or orig_id not in mapping_results:
        continue
        
    full_seq = fasta_sequences[fasta_id]
    info = mapping_results[orig_id]
    
    parser = MMCIFParser(QUIET=True) if info['type'] == "CIF" else PDBParser(QUIET=True)
    try:
        struct = parser.get_structure('temp', info['file'])
        chain_obj = struct[0][row['PDB_CHAIN_MAPPED']]
        pdb_seq, pdb_res_ids = get_pdb_seq_and_map(chain_obj)
        
        # Global Alignment 수행
        alignments = pairwise2.align.globalms(full_seq, pdb_seq, 2, -1, -0.5, -0.1)
        
        if alignments:
            best_aln = alignments[0]
            aligned_full, aligned_pdb, score, start, end = best_aln
            
            # FASTA POS (1-based)를 aligned 서열의 인덱스로 변환
            target_pos_1based = int(row['POS'])
            current_fasta_idx = 0
            mapped_idx_in_aln = -1
            
            for i, char in enumerate(aligned_full):
                if char != '-':
                    current_fasta_idx += 1
                if current_fasta_idx == target_pos_1based:
                    mapped_idx_in_aln = i
                    break
            
            # aligned_pdb에서 해당 위치의 실제 PDB 잔기 인덱스 찾기
            if mapped_idx_in_aln != -1 and aligned_pdb[mapped_idx_in_aln] != '-':
                # '-'를 제외하고 pdb_seq에서 몇 번째 아미노산인지 계산
                pdb_idx = len(aligned_pdb[:mapped_idx_in_aln].replace('-', ''))
                real_pdb_res_id = pdb_res_ids[pdb_idx]
                
                # 검증: 아미노산이 일치하는지
                if aligned_pdb[mapped_idx_in_aln] == row['WT']:
                    df.at[idx, 'PDB_POS_MAPPED'] = real_pdb_res_id
                    df.at[idx, 'PDB_WT_MAPPED'] = aligned_pdb[mapped_idx_in_aln]
                    df.at[idx, 'MAPPING_STATUS'] = "Aligned_Success"
                else:
                    df.at[idx, 'MAPPING_STATUS'] = "Aligned_WT_Mismatch"
            else:
                df.at[idx, 'MAPPING_STATUS'] = "Not_In_Structure"
                
    except Exception as e:
        df.at[idx, 'MAPPING_STATUS'] = f"Align_Error: {str(e)}"

print("\n" + "="*30)
print(df['MAPPING_STATUS'].value_counts())
print("="*30)

df.to_csv("S2450_alignment_fixed.tsv", sep='\t', index=False)

FASTA 기반 정밀 Alignment 매핑 시작...

MAPPING_STATUS
Aligned_Success       2121
Not_In_Structure       163
Align_Error: 'OCS'     102
Align_Error: 'CSD'      43
Align_Error: 'KST'      14
Align_Error: 'PCA'       3
Align_Error: 'TRN'       2
Failed                   2
Name: count, dtype: int64
